In [1]:
from google.colab import drive
drive.mount('/content/drive')

#!wget https://zenodo.org/record/835510/files/part1_cropped.tar.gz -O /content/drive/MyDrive/icub_data/part1_cropped.tar.gz


Mounted at /content/drive


In [2]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 43.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/google/vit-base-patch16-224

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/vit-base-patch16-224)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [3]:
# Load model directly
#from transformers import AutoImageProcessor, AutoModelForImageClassification

#processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
#model = AutoModelForImageClassification.from_pretrained("google/vit-base-patch16-224")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Rudimentary test

In [4]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import requests
import torch

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits
##  Model predicts one of the 1000 ImageNet classes.
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Predicted class: Egyptian cat


Extract downloaded iCub dataset.

In [10]:

!rm -r /content/all_images /content/icub_dataset

!mkdir /content/all_images
!mkdir /content/icub_dataset
!ls /content/drive/MyDrive/modules_work/COMP34212/icub_data
!tar xf /content/drive/MyDrive/modules_work/COMP34212/icub_data/part1_cropped.tar.gz -C /content/icub_dataset/

part1_cropped.tar.gz  part3_cropped.tar.gz
part2_cropped.tar.gz  part4_cropped.tar.gz


In [12]:
import glob
import os
import shutil
import pandas as pd

images = glob.glob("/content/icub_dataset/**/*.jpg", recursive=True)

monolith_dir = "/content/all_images"

metadata = []


for img in images:

  #print(img)

  file_path_split = img.split('/')

  #print(file_path_split)

  label = file_path_split[4]

  filename = file_path_split[-1]

  new_filename = f"{label}_{filename}"

  print(new_filename)

  metadata = metadata + [{
      "file_name": new_filename,
      "label": label
  }]

  shutil.move(img, os.path.join(monolith_dir, new_filename))


In [13]:
!find /content/icub_dataset -iname "*jpg" -exec mv -t /content/all_images/ {} ';'

In [14]:
##  Remove old metadata.

!rm /content/all_images/metadata.csv

##  Write new metadata.

df = pd.DataFrame(metadata)

metadata_csv = os.path.join(monolith_dir, "metadata.csv")

include_header = not os.path.exists(metadata_csv)

df.to_csv(metadata_csv,
          mode='a',
          index=False,
          header=include_header)

rm: cannot remove '/content/all_images/metadata.csv': No such file or directory


In [15]:
#!sed -i 's/filename/file_name/g' /content/all_images/metadata.csv

!head /content/all_images/metadata.csv

!tail /content/all_images/metadata.csv

file_name,label
00002499.jpg,pencilcase
00002642.jpg,pencilcase
00002511.jpg,pencilcase
00002498.jpg,pencilcase
00002671.jpg,pencilcase
00002673.jpg,pencilcase
00002570.jpg,pencilcase
00002660.jpg,pencilcase
00002624.jpg,pencilcase
00002666.jpg,ringbinder
00002661.jpg,ringbinder
00002439.jpg,ringbinder
00002623.jpg,ringbinder
00002412.jpg,ringbinder
00002504.jpg,ringbinder
00002432.jpg,ringbinder
00002520.jpg,ringbinder
00002416.jpg,ringbinder
00002390.jpg,ringbinder


In [16]:
from datasets import load_dataset, ClassLabel

dataset = load_dataset("imagefolder", data_dir="/content/all_images/")

print(dataset["train"][0])

dataset["train"] = dataset["train"].class_encode_column("label")

Resolving data files:   0%|          | 0/9748 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2745C6DEE0>, 'label': 'pencilcase'}


Casting to class labels:   0%|          | 0/194918 [00:00<?, ? examples/s]

In [17]:
print(dataset["train"][0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D276834F200>, 'label': 3}


In [19]:
ds_split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=6948566879846,
    stratify_by_column="label"
)

####  **iCub World dataset categories**

PART1: book, cellphone, mouse, pencilcase, ringbinder.

PART2: hairbrush, hairclip, perfume, sunglasses, wallet.

PART3: flower, glass, mug, remote, soapdispenser.

PART4: bodylotion, ovenglove, sodabottle, sprayer, squeezer.

####  **ImageNet dataset categories**

ImageNet categories that model was trained on:

https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/

Deduce mapping between pretrained labels and dataset labels.

In [20]:
label_rename = {
    "pencilcase": "pencil case",
    "ringbinder": "ring binder",
    "soapdispenser": "dispenser",
    "bodylotion": "lotion",
    "ovenglove": "glove",
    "sodabottle": "soda",
    "mouse": "computer mouse"
    }

model_id_label = model.config.id2label

model_label_id = {v: k for k, v in model.config.id2label.items()}

ds_labels = dataset["train"].features["label"].names

print(ds_labels)

ds_labels = tuple(map(
    lambda label: label_rename[label] if label in label_rename.keys() else label,
    ds_labels
))


print(ds_labels)

ds_id_label = {i: label for i, label in enumerate(ds_labels)}

ds_label_id = {label: i for i, label in ds_id_label.items()}

print(model_id_label)
print(model_label_id)
print(ds_id_label)
print(ds_label_id)

ds_model_id_map = {}

ds_model_label_map = {}

for id, label in ds_id_label.items():

  for label_match in tuple(filter(lambda x: label in x, model_label_id.keys())):

    ds_model_label_map[label] = label_match

    ds_model_id_map[id] = model_label_id[label_match]


print(ds_model_id_map)
print(ds_model_label_map)

['book', 'cellphone', 'mouse', 'pencilcase', 'ringbinder']
('book', 'cellphone', 'computer mouse', 'pencil case', 'ring binder')
{0: 'tench, Tinca tinca', 1: 'goldfish, Carassius auratus', 2: 'great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias', 3: 'tiger shark, Galeocerdo cuvieri', 4: 'hammerhead, hammerhead shark', 5: 'electric ray, crampfish, numbfish, torpedo', 6: 'stingray', 7: 'cock', 8: 'hen', 9: 'ostrich, Struthio camelus', 10: 'brambling, Fringilla montifringilla', 11: 'goldfinch, Carduelis carduelis', 12: 'house finch, linnet, Carpodacus mexicanus', 13: 'junco, snowbird', 14: 'indigo bunting, indigo finch, indigo bird, Passerina cyanea', 15: 'robin, American robin, Turdus migratorius', 16: 'bulbul', 17: 'jay', 18: 'magpie', 19: 'chickadee', 20: 'water ouzel, dipper', 21: 'kite', 22: 'bald eagle, American eagle, Haliaeetus leucocephalus', 23: 'vulture', 24: 'great grey owl, great gray owl, Strix nebulosa', 25: 'European fire salamander, Salaman

Run all data through model and check for accurate predictions, calculate F1 score.

In [21]:
model_ds_id_map = {label: i for i, label in ds_model_id_map.items()}

print(model_ds_id_map)

##  Reserve class 0 for y_hats that do not match any class.

n = len(ds_id_label.keys()) + 1

##  m = len(model_id_label.keys())

f1_matrix = [[0 for _ in range(n)] for _ in range(n)]

print(f1_matrix)

image_benchmark = Image.open(requests.get(url, stream=True).raw)

print(image_benchmark)

print(ds_split["test"])

y_hat_set = []

y_set = []

{921: 0, 487: 1, 673: 2, 709: 3}
[[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]]
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=640x480 at 0x7D2744DC8D10>
Dataset({
    features: ['image', 'label'],
    num_rows: 38984
})


## **Experiment #1**

In [22]:
def test_model(model, testset, ds_model_id_map, model_ds_id_map, n):

  y_set, y_hat_set = [], []

  x = 0

  for img in testset:

    if x == 1000:

      break

    print(img)

    y = img["label"]

    if y in ds_model_id_map.keys():

      x += 1

      ##  print(ds_model_id_map[y])

      inputs = processor(img["image"], return_tensors = "pt")

      outputs = model(**inputs)

      logits = outputs.logits

      predicted_class_idx = logits.argmax(-1).item()

      ##  Convert to y_hat via dataset category id.

      if predicted_class_idx not in model_ds_id_map.keys():

          y_hat = n - 1

      else:

          y_hat = model_ds_id_map[predicted_class_idx]

      ##  f1_matrix[y_hat][y] += 1

      y_hat_set = y_hat_set + [y_hat]

      y_set = y_set + [y]

      print(f"{x}\t{y}\t{y_hat}")

  return y_set, y_hat_set

In [23]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

def calc_f1(y_set: list[int], y_hat_set: list[int]):

  f1 = f1_score(y_set, y_hat_set, average=None)

  f1_weighted = f1_score(y_set, y_hat_set, average="weighted")

  print(f1)

  print(f1_weighted)

  return f1, f1_weighted

New dataset, pretrained model, no finetuning.

In [24]:
experiment_1_done = True

if experiment_1_done == False:

  y_set, y_hat_set = test_model(model,
                                ds_split["test"],
                                ds_model_id_map,
                                model_ds_id_map,
                                len(ds_id_label.keys()) + 1)

  calc_f1(y_set, y_hat_set)

## **Experiment #2**

New dataset, pretrained model, finetuning experimenting.

In [25]:
print(dataset["train"][0])
print(dataset["train"])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FD4EC0>, 'label': 3}
Dataset({
    features: ['image', 'label'],
    num_rows: 194918
})


In [48]:
def create_iterator(start_val):

  counter = start_val

  def incrementer():
      nonlocal counter
      counter += 1
      return counter

  return incrementer


def transform_img(examples):

  ##  print(examples)

  print(train_iter())

  ##  Convert image format to tensor format, accepted by training process.

  inputs = processor(
      [img.convert("RGB") for img in examples["image"]],
      return_tensors = "pt"
      )

  inputs["labels"] = examples["label"]

  return inputs


def transform_label(examples):

  ##  print('examples', examples)

  ##  Convert labels to ImageNet equivalents.

  new_labels = list(map(
      lambda l: ds_model_id_map[l] if l in ds_model_id_map else 1,
      examples["label"]
  ))

  ##  print(examples["label"])

  ##  print(new_labels)

  inputs["labels"] = new_labels

  return inputs

Filter iCub World dataset to retrieve only data points for which a corresponding label exists in the original ImageNet dataset the model was trained on. This prevents the training of data for which no corresponding class exists.

In [28]:
#ds_subset_filtered = dataset_filtered["train"].train_test_split(
#  test_size = 0.2,
#  stratify_by_column = "label"
#  seed = 6948566879846,
#)

##  Trains OK.
#ds_subset_train_img = ds_subset["train"].with_transform(transform_img)
#ds_subset_test_img = ds_subset["test"].with_transform(transform_img)

##  Trains OK.
#ds_subset_filt_train_img = ds_subset_filtered["train"].with_transform(transform_img)
#ds_subset_filt_test_img = ds_subset_filtered["test"].with_transform(transform_img)

In [27]:
dataset_filtered = dataset.filter(lambda d: d["label"] in ds_model_id_map.keys())

Filter:   0%|          | 0/194918 [00:00<?, ? examples/s]

                                                                               image  \
0  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB4890>   
1  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB56A0>   
2  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB5400>   
3  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB77D0>   
4  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB5BB0>   
5  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB7B60>   
6  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB54C0>   
7  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB7E00>   
8  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB6000>   
9  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779FB7770>   

   label  
0      3  
1      3 

Adjust the labels of the data to match the ImageNet dataset.

Also run the images through the image processor.

In [40]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print(pd.DataFrame(dataset_filtered["train"].shuffle(seed=3954385)[:10]))

print(dataset_filtered)

                                                                               image  \
0  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C15D30>   
1  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C15700>   
2  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C14170>   
3  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C17740>   
4  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C173B0>   
5  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C17560>   
6  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C17D70>   
7  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C165A0>   
8  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C16900>   
9  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=256x256 at 0x7D2779C17A10>   

   label  
0      3  
1      3 

In [42]:
ds_subset_filtered = dataset_filtered["train"].train_test_split(
  #train_size = 20,
  test_size = 0.2,
  seed = 6948566879846,
  stratify_by_column = "label"
)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 159853
    })
})


In [52]:
##  OK as of 19/03/2026 10:41am.
ds_subset_filt_train_img_relabel = ds_subset_filtered["train"].with_transform(transform_img).with_transform(transform_label)

##  Split again for validation and test sets.
ds_subset_filt_testvalid_img_relabel = ds_subset_filtered["test"].train_test_split(
  train_size = 0.5,
  test_size = 0.5,
  seed = 6948566879846,
  stratify_by_column = "label"
)

ds_subset_filt_validation_img_relabel = ds_subset_filt_testvalid_img_relabel["train"].with_transform(transform_img).with_transform(transform_label)

print(dataset_filtered)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 159853
    })
})


In [82]:
print(ds_subset_filt_train_img_relabel)

Dataset({
    features: ['image', 'label'],
    num_rows: 127882
})


Fine-tune model. Note that the HuggingFace library will default to GPU.

In [53]:
from transformers import TrainingArguments, Trainer

import torch

##  print(ds_split["train"][0])

train_iter = create_iterator(0)

model_ex_2 = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224'
)

training_args = TrainingArguments(
    output_dir = "/content/model_ex_2",
    num_train_epochs = 8,               ##  Frequency of times to feed training data.
    per_device_train_batch_size = 256,  ##  Batch size when training.
    gradient_accumulation_steps = 8,    ##  Delay before backpropagation.
    gradient_checkpointing = True,      ##  More on-the-fly during backpropagation.
    learning_rate = 2e-5,               ##  The rate of parameter iteration.
    logging_steps = 10,                 ##  How often to update loss function.
    eval_strategy = "epoch",            ##  Run evaluation at end of epoch.
    save_strategy = "epoch",            ##  Save at end of each epoch.
    load_best_model_at_end = True,      ##  Load best model at end, via evaluations.
    remove_unused_columns = False,
)

trainer = Trainer(
    model = model_ex_2,
    args = training_args,
    train_dataset = ds_subset_filt_train_img_relabel,
    eval_dataset = ds_subset_filt_validation_img_relabel,
    ##  processing_class = tokenizer,
    ##  data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Training with device:", trainer.args.device)

experiment_2_train_done = True

if experiment_2_train_done == False:

  trainer.train()

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Training with device: cuda:0


Epoch,Training Loss,Validation Loss
1,1.700015,1.597195
2,1.442027,1.431958
3,1.543859,1.423159
4,1.442388,1.484489
5,1.436952,1.399872
6,1.470906,1.410491
7,1.410272,1.395463
8,1.365876,1.409290


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=504, training_loss=1.5402600528701904, metrics={'train_runtime': 1392.3155, 'train_samples_per_second': 734.787, 'train_steps_per_second': 0.362, 'total_flos': 3.12740546936832e+17, 'train_loss': 1.5402600528701904, 'epoch': 8.0})

In [ ]:
import matplotlib.pyplot as plt

# 1. Convert log history to a Pandas DataFrame
history = pd.DataFrame(trainer.state.log_history)

# 2. Separate training and evaluation logs
# Training logs usually have 'loss', eval logs have 'eval_loss'
train_loss = history[history['loss'].notna()]
eval_loss = history[history['eval_loss'].notna()]

plt.figure(figsize=(10, 5))

# Plotting Training Loss
plt.plot(train_loss['epoch'], train_loss['loss'], label='Training Loss', marker='o')

# Plotting Evaluation Loss
plt.plot(eval_loss['epoch'], eval_loss['eval_loss'], label='Evaluation Loss', marker='s')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()
plt.savefig("ex_2_loss_func.pdf", format="pdf")

In [58]:
def transform_label_v2(examples):

  new_labels = list(map(
      lambda l: ds_model_id_map[l] if l in ds_model_id_map else 1,
      examples["label"]
  ))

  updated_examples = {
      "labels": new_labels,
      "image": examples["image"]
  }

  if 'pixel_values' in examples:
      updated_examples['pixel_values'] = examples['pixel_values']

  return updated_examples


##print(pd.DataFrame(ds_subset_filt_test_img_relabel_v3[:10]))

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 159853
    })
})


In [62]:
from transformers import pipeline

##  print(ds_model_id_map)


def data_generator(testset):
    for sample in testset:
        # For ViT, we yield the PIL image object directly
        yield sample["image"]


def test_model_ex_2(m_classifier, processor, testset):

  y_set, y_hat_set = [], []

  pipe = pipeline(
      "image-classification",
      model = m_classifier,
      image_processor = processor,
      torch_dtype = torch.float16
      )

  for i, out in enumerate(pipe(data_generator(testset), batch_size=256)):

    y_hat_label = out[0]['label']

    y_hat = pipe.model.config.label2id[y_hat_label]

    y = testset[i]['labels']

    print(i)

    ##  print(y_hat, y)

    print(y, y_hat)

    y_set = y_set + [y]

    y_hat_set = y_hat_set + [y_hat]

  return y_set, y_hat_set


ds_subset_filt_test_img_relabel = ds_subset_filt_testvalid_img_relabel["test"].with_transform(transform_label_v2)

print(dataset_filtered)


experiment_2_test_done = True

if experiment_2_test_done == False:

  y_set, y_hat_set = test_model_ex_2(
      model_ex_2,
      ViTImageProcessor.from_pretrained('google/vit-base-patch16-224'),
      ds_subset_filt_test_img_relabel
      )

  f1_each, f1_weighted = calc_f1(y_set, y_hat_set)

Streaming output truncated to the last 5000 lines.
13503
673 446
13504
709 446
13505
709 446
13506
709 673
13507
709 610
13508
673 446
13509
921 446
13510
487 446
13511
709 527
13512
673 834
13513
487 446
13514
487 667
13515
921 587
13516
921 681
13517
673 446
13518
921 788
13519
709 893
13520
921 620
13521
673 593
13522
487 446
13523
487 446
13524
709 624
13525
921 673
13526
709 860
13527
487 446
13528
921 624
13529
487 673
13530
709 446
13531
921 527
13532
709 446
13533
921 624
13534
709 446
13535
673 446
13536
709 446
13537
673 446
13538
487 446
13539
921 526
13540
709 624
13541
487 590
13542
921 624
13543
487 527
13544
709 542
13545
921 446
13546
487 921
13547
673 620
13548
709 681
13549
487 446
13550
487 620
13551
921 446
13552
487 921
13553
487 549
13554
709 673
13555
673 542
13556
921 673
13557
709 549
13558
487 624
13559
673 620
13560
709 527
13561
487 921
13562
487 699
13563
709 921
13564
921 446
13565
673 446
13566
921 709
13567
487 673
13568
487 620
13569
921 673
13570
673 6

In [63]:
f1_each, f1_weighted = calc_f1(y_set, y_hat_set)

report = classification_report(
    y_set,
    y_hat_set,
    output_dict=False
)

print(report)

[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.00504909 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.16431188 0.         0.         0.         0.         

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#print(next(model.parameters()).dtype)
#for name, param in model.named_parameters():
#    print(f"Layer: {name} | Dtype: {param.dtype} | Device: {param.device}")

print(next(model_ex_2.parameters()).dtype)
for name, param in model_ex_2.named_parameters():
    print(f"Layer: {name} | Dtype: {param.dtype} | Device: {param.device}")

## **Experiment #3**

Fine-tuning class with best F1 score.

In [70]:
ds_673_filtered = dataset.filter(lambda d: d["label"] == model_ds_id_map[673])

In [71]:
ds_673_first_split = ds_673_filtered["train"].train_test_split(
  test_size = 0.2,
  seed = 6948566879846,
  stratify_by_column = "label"
)

ds_673_train = ds_673_first_split["train"].with_transform(transform_img).with_transform(transform_label)

##  Split again for validation and test sets.

ds_673_second_split = ds_673_first_split["test"].train_test_split(
  train_size = 0.5,
  test_size = 0.5,
  seed = 6948566879846,
  stratify_by_column = "label"
)

ds_673_validation = ds_673_second_split["train"].with_transform(transform_img).with_transform(transform_label)

ds_673_test = ds_673_second_split["test"].with_transform(transform_label_v2)

In [ ]:
print(ds_673_filtered)
print(ds_673_train)
print(ds_673_test)

print(pd.DataFrame(ds_673_first_split["train"].shuffle(seed=3954385)[:10]))

print(pd.DataFrame(ds_673_second_split["train"].shuffle(seed=3954385)[:10]))

In [ ]:
train_iter = create_iterator(0)

model_ex_3 = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224'
)

training_args_ex_3 = TrainingArguments(
    output_dir = "/content/model_ex_3",
    num_train_epochs = 32,               ##  Frequency of times to feed training data.
    per_device_train_batch_size = 16,   ##  Batch size when training.
    per_device_eval_batch_size = 4,     ##  Batch size when training.
    gradient_accumulation_steps = 32,   ##  Delay before backpropagation.
    gradient_checkpointing = True,      ##  More on-the-fly during backpropagation.
    learning_rate = 2e-5,               ##  The rate of parameter iteration.
    logging_steps = 10,                 ##  How often to update loss function.
    eval_strategy = "epoch",            ##  Run evaluation at end of epoch.
    save_strategy = "epoch",            ##  Save at end of each epoch.
    load_best_model_at_end = True,      ##  Load best model at end, via evaluations.
    remove_unused_columns = False,
)



trainer_ex_3 = Trainer(
    model = model_ex_3,
    args = training_args_ex_3,
    train_dataset = ds_673_train,
    eval_dataset = ds_673_validation,
    ##  processing_class = tokenizer,
    ##  data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Training with device:", trainer.args.device)

trainer_ex_3.train()

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Training with device: cuda:0


Epoch,Training Loss,Validation Loss
1,0.001985,0.001586
2,0.000813,0.000748
3,0.000489,0.000471
4,0.000360,0.000339
5,0.000273,0.000263
6,0.000218,0.000213
7,0.000185,0.000179
8,0.000158,0.000154
9,0.000138,0.000135
10,0.000122,0.000121


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## **Experiment #4**

Fine-tuning class with worst F1 score.

In [ ]:
dataset_filtered_best_f1 = dataset.filter(lambda d: d["label"] == model_ds_id_map[709])